# EvaluatorAgent - 動作確認用ノートブック

## 概要

EvaluatorAgent は、テキストや URL などの評価対象に対して多角的な評価を行うエージェントです。

### ワークフロー

1. **GoalCreator** - 評価目標・基準を策定
2. **Planner** - 評価計画を立案
3. **Executor** - 情報収集・評価を実行
4. **Reflector** - 評価の十分性を検証（不十分なら Executor に戻る、最大5回）
5. **Integrator** - 結果を統合し最終出力を生成

### 入出力

- **入力**: `EvaluationInput` - 評価対象、対象種類、評価リクエスト、追加コンテキスト
- **出力**: `EvaluationOutput` - 総合スコア、基準別スコア、長所・短所・改善点、サマリー

In [ ]:
# 環境セットアップ
import subprocess
import sys
from pathlib import Path

# tools/ ディレクトリを特定
notebook_dir = Path.cwd()
for candidate in [
    notebook_dir.parents[3],  # notebook → evaluator → agents → src → tools
    notebook_dir.parents[4],
    notebook_dir,
]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        tools_dir = str(candidate)
        break
else:
    tools_dir = str(notebook_dir)

if tools_dir not in sys.path:
    sys.path.insert(0, tools_dir)

print(f"Python: {sys.executable}")
print(f"Tools directory: {tools_dir}")

# python-dotenv がなければインストール
try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv

load_dotenv(Path(tools_dir) / ".env")
print("環境変数を読み込みました")

## 入力データの説明

`EvaluationInput` は以下のフィールドを持ちます。

| フィールド | 型 | 説明 |
|---|---|---|
| `target` | `str` | 評価対象（テキストまたは URL） |
| `target_type` | `"article"` \| `"service"` \| `"product"` \| `"other"` | 評価対象の種類（デフォルト: `"article"`） |
| `evaluation_request` | `str` | 評価してほしい内容の説明 |
| `context` | `str \| None` | 追加コンテキスト（任意） |

In [ ]:
# 入力データの作成
from src.agents.evaluator.schemas import EvaluationInput

input_data = EvaluationInput(
    target="評価対象のテキストまたはURL",
    target_type="article",
    evaluation_request="SEO観点で評価してください",
    context=None,
)

print(f"評価対象: {input_data.target}")
print(f"対象種類: {input_data.target_type}")
print(f"評価リクエスト: {input_data.evaluation_request}")
print(f"追加コンテキスト: {input_data.context}")

## エージェント実行

`EvaluatorAgent` をインスタンス化し、`run()` メソッドで実行します。

> **注意**: 実行すると LLM API が呼び出されます。`OPENAI_API_KEY` が `.env` に設定されていることを確認してください。

In [ ]:
# エージェントをインスタンス化して実行
from src.agents.evaluator.agent import EvaluatorAgent

agent = EvaluatorAgent()
result = agent.run(input_data)

print("評価が完了しました")

## 結果の確認

`EvaluationOutput` の内容を見やすくフォーマットして表示します。

In [ ]:
# 結果をフォーマットして表示
print("=" * 60)
print("評価結果")
print("=" * 60)
print(f"対象の要約: {result.target_summary}")
print(f"対象種類:   {result.target_type}")
print(f"総合スコア: {result.overall_score} / 100")
print()

if result.criterion_scores:
    print("-" * 40)
    print("基準別スコア")
    print("-" * 40)
    for cs in result.criterion_scores:
        print(f"  [{cs.score:3d}] {cs.criterion}")
        print(f"        {cs.rationale}")
    print()

if result.strengths:
    print("-" * 40)
    print("長所")
    print("-" * 40)
    for s in result.strengths:
        print(f"  + {s}")
    print()

if result.weaknesses:
    print("-" * 40)
    print("短所")
    print("-" * 40)
    for w in result.weaknesses:
        print(f"  - {w}")
    print()

if result.improvements:
    print("-" * 40)
    print("改善点")
    print("-" * 40)
    for i, imp in enumerate(result.improvements, 1):
        print(f"  {i}. {imp}")
    print()

print("=" * 60)
print("サマリー")
print("=" * 60)
print(result.summary)